# Streaming

<div class="alert alert-success">

Đối với các ứng dụng mới, chúng tôi khuyên dùng [event streaming](https://docs.langchain.com/oss/python/langchain/event-streaming) - API typed-projection được giới thiệu trong LangChain v1.3. Event streaming cung cấp cho bạn các iterator riêng biệt cho từng phép chiếu (message, value, tool call, subgraph) để bạn có thể sử dụng chúng một cách độc lập thay vì phải phân nhánh trên các chunk của `stream_mode`.

</div>

LangChain triển khai một hệ thống streaming để hiển thị các bản cập nhật theo thời gian thực.

Streaming đóng vai trò quan trọng trong việc nâng cao tốc độ phản hồi của các ứng dụng xây dựng trên LLM. Bằng cách hiển thị đầu ra một cách tăng dần, ngay cả khi chưa có phản hồi hoàn chỉnh, streaming cải thiện đáng kể trải nghiệm người dùng (UX), đặc biệt là khi phải xử lý độ trễ của các LLM.

## Tổng quan

Hệ thống streaming của LangChain cho phép bạn hiển thị phản hồi trực tiếp từ các lượt chạy agent lên ứng dụng của mình.

Những gì có thể làm với streaming của LangChain:

* [**Stream tiến trình của agent**](https://docs.langchain.com/oss/python/langchain/streaming#agent-progress) - nhận các bản cập nhật trạng thái sau mỗi bước của agent.
* [**Stream các token của LLM**](https://docs.langchain.com/oss/python/langchain/streaming#llm-tokens) - stream các token của mô hình ngôn ngữ ngay khi chúng được tạo ra.
* [**Stream các token tư duy / suy luận**](https://docs.langchain.com/oss/python/langchain/streaming#streaming-thinking-/-reasoning-tokens) - hiển thị quá trình suy luận của mô hình ngay khi nó được tạo ra.
* [**Stream các bản cập nhật tùy chỉnh**](https://docs.langchain.com/oss/python/langchain/streaming#custom-updates) - phát ra các tín hiệu do người dùng định nghĩa (ví dụ: `"Đã tải 10/100 bản ghi"`).
* [**Stream nhiều chế độ**](https://docs.langchain.com/oss/python/langchain/streaming#stream-multiple-modes) - chọn từ `updates` (tiến trình của agent), `messages` (token của LLM + metadata), hoặc `custom` (dữ liệu bất kỳ của người dùng).

Xem phần [các pattern phổ biến](https://docs.langchain.com/oss/python/langchain/streaming#common-patterns) bên dưới để biết thêm các ví dụ thực tế.

## Các chế độ stream được hỗ trợ

Truyền một hoặc nhiều chế độ stream dưới đây dưới dạng một danh sách vào các phương thức [`stream`](https://reference.langchain.com/python/langgraph/graphs/#langgraph.graph.state.CompiledStateGraph.stream) hoặc [`astream`](https://reference.langchain.com/python/langgraph/graphs/#langgraph.graph.state.CompiledStateGraph.astream):

| Chế độ     | Mô tả                                                                                                                                                                             |
| ---------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `updates`  | Stream các bản cập nhật trạng thái sau mỗi bước của agent. Nếu có nhiều bản cập nhật được thực hiện trong cùng một bước (ví dụ: nhiều node chạy cùng lúc), các bản cập nhật đó sẽ được stream riêng biệt. |
| `messages` | Stream các tuple dạng `(token, metadata)` từ bất kỳ node nào trong graph nơi LLM được gọi.                                                                                        |
| `custom`   | Stream dữ liệu tùy chỉnh từ bên trong các node của graph bằng cách sử dụng stream writer.                                                                                         |

## Tiến trình của agent

Để stream tiến trình của agent, hãy sử dụng các phương thức [`stream`](https://reference.langchain.com/python/langgraph/graphs/#langgraph.graph.state.CompiledStateGraph.stream) hoặc [`astream`](https://reference.langchain.com/python/langgraph/graphs/#langgraph.graph.state.CompiledStateGraph.astream) với `stream_mode="updates"`. Điều này sẽ phát ra một event sau mỗi bước của agent.

Ví dụ: nếu bạn có một agent gọi một tool một lần, bạn sẽ thấy các bản cập nhật sau:

* **Node LLM**: [`AIMessage`](https://reference.langchain.com/python/langchain-core/messages/ai/AIMessage) chứa các yêu cầu tool call
* **Node Tool**: [`ToolMessage`](https://reference.langchain.com/python/langchain-core/messages/tool/ToolMessage) chứa kết quả thực thi
* **Node LLM**: Phản hồi cuối cùng của AI

Truyền một `thread_id` thông qua `config` để cuộc trò chuyện được checkpoint và các lượt hội thoại tiếp theo có thể tiếp tục với cùng một lịch sử. `thread_id` độc lập với `stream_mode`; bạn cũng có thể truyền `context` cùng với nó để lưu trữ dữ liệu cho từng lượt chạy mà các tool của bạn có thể đọc từ `runtime.context`.

In [2]:
from langchain.agents import create_agent
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver

def get_weather(city: str) -> str:
    """Lấy thông tin thời tiết cho một thành phố nhất định."""
    return f"Trời luôn nắng ở {city}!"

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[get_weather],
    checkpointer=InMemorySaver()
)
config = {"configurable": {"thread_id": str(uuid7())}}
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "Thời tiết ở SF như thế nào?"}]},
    config=config,
    version="v3",
)
for kind, item in stream.interleave("messages", "tool_calls"):
    if kind == "messages":
        for token in item.text:
            print(token, end="", flush=True)
    elif kind == "tool_calls":
        print(f"\nGọi tool: {item.tool_name}({item.input})")
        for delta in item.output_deltas:
            print(delta, end="", flush=True)
        print(f"\nKết quả tool: {item.output}")

final_state = stream.output


Gọi tool: get_weather({'city': 'SF'})

Kết quả tool: content='Trời luôn nắng ở SF!' name='get_weather' tool_call_id='call_643591'
Thời tiết ở SF hiện tại: Trời luôn nắng ở SF!

<div class="alert alert-info">

Việc duy trì lịch sử trò chuyện bằng `thread_id` yêu cầu agent phải được cấu hình với một [checkpointer](https://docs.langchain.com/oss/python/langchain/long-term-memory). Trên các bản triển khai [LangSmith](https://docs.langchain.com/langsmith/deployment), một checkpointer sẽ được cung cấp tự động. Khi chạy local, hãy truyền nó một cách tường minh, ví dụ: `create_agent(..., checkpointer=InMemorySaver())`. Các đoạn mã còn lại trên trang này sẽ bỏ qua `thread_id` cho ngắn gọn, nhưng bạn nên truyền nó khi chạy trên production.

</div>

## Token của LLM

Để stream các token ngay khi chúng được tạo ra bởi LLM, hãy sử dụng `stream_mode="messages"`. Dưới đây, bạn có thể xem đầu ra của agent khi stream các tool call và phản hồi cuối cùng.

In [3]:
from langchain.agents import create_agent


def get_weather(city: str) -> str:
    """Lấy thông tin thời tiết cho một thành phố nhất định."""

    return f"Trời luôn nắng ở {city}!"

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[get_weather],
)
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "Thời tiết ở SF như thế nào?"}]},
    stream_mode="messages",
    version="v2",
):
    if chunk["type"] == "messages":
        token, metadata = chunk["data"]
        print(f"node: {metadata['langgraph_node']}")
        print(f"nội dung: {token.content_blocks}")
        print("\n")

node: model
nội dung: [{'type': 'tool_call', 'id': 'call_157497', 'name': 'get_weather', 'args': {'city': 'SF'}}]


node: model
nội dung: []


node: model
nội dung: []


node: tools
nội dung: [{'type': 'text', 'text': 'Trời luôn nắng ở SF!'}]


node: model
nội dung: [{'type': 'text', 'text': 'Thời tiết', 'index': 0}]


node: model
nội dung: [{'type': 'text', 'text': ' ở SF hiện tại là: Trời luôn nắng ở SF!', 'index': 0}]


node: model
nội dung: [{'type': 'text', 'text': '', 'extras': {'signature': 'El4KXAERTTIPfHOLHiB2pp9yJgzMhvkv8xOfwZDL+mPStGjWGIWLPKTmYWoLJTlOlJwSXjOvs4pfLa/WcTpZzlOXOHFovy9oPtq6cWE07bNq9VRG9pVkrE/I82Ic0b+E'}, 'index': 0}]


node: model
nội dung: []




<div class="alert alert-info">

**Bao bọc một agent dưới dạng một node trong một `StateGraph` cha?** [`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_agent) trả về một graph đã được biên dịch, vì vậy việc sử dụng nó như một node sẽ biến nó thành một subgraph. Việc thiết lập `stream_mode="messages"` trên graph cha sẽ không phát ra các token chunk từ các lệnh gọi LLM của agent nội bộ trừ khi bạn truyền `subgraphs=True`. Xem [Đầu ra của Subgraph](https://docs.langchain.com/oss/python/langgraph/streaming#subgraph-outputs).

</div>

## Các bản cập nhật tùy chỉnh

Để stream các bản cập nhật từ các tool khi chúng đang được thực thi, bạn có thể sử dụng [`get_stream_writer`](https://reference.langchain.com/python/langgraph/config/get_stream_writer).

In [4]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_weather(city: str) -> str:
    """Lấy thông tin thời tiết cho một thành phố nhất định."""
    writer = get_stream_writer()
    # stream dữ liệu bất kỳ
    writer(f"Đang tra cứu dữ liệu cho thành phố: {city}")
    writer(f"Đã thu thập dữ liệu cho thành phố: {city}")
    return f"Trời luôn nắng ở {city}!"

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "Thời tiết ở SF như thế nào?"}]},
    stream_mode="custom",
    version="v2",
):
    if chunk["type"] == "custom":
        print(chunk["data"])

Đang tra cứu dữ liệu cho thành phố: San Francisco
Đã thu thập dữ liệu cho thành phố: San Francisco


<div class="alert alert-info">

Nếu bạn thêm [`get_stream_writer`](https://reference.langchain.com/python/langgraph/config/get_stream_writer) vào bên trong tool của mình, bạn sẽ không thể gọi tool đó bên ngoài ngữ cảnh thực thi của LangGraph.

</div>

## Stream nhiều chế độ

Bạn có thể chỉ định nhiều chế độ stream bằng cách truyền stream mode dưới dạng một danh sách: `stream_mode=["updates", "custom"]`.

Mỗi chunk được stream là một dict `StreamPart` chứa các key `type`, `ns` và `data`. Hãy sử dụng `chunk["type"]` để xác định chế độ stream và `chunk["data"]` để truy cập dữ liệu payload.

In [5]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_weather(city: str) -> str:
    """Lấy thông tin thời tiết cho một thành phố nhất định."""
    writer = get_stream_writer()
    writer(f"Đang tra cứu dữ liệu cho thành phố: {city}")
    writer(f"Đã thu thập dữ liệu cho thành phố: {city}")
    return f"Trời luôn nắng ở {city}!"

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "Thời tiết ở SF như thế nào?"}]},
    stream_mode=["updates", "custom"],
    version="v2",
):
    print(f"stream_mode: {chunk['type']}")
    print(f"nội dung: {chunk['data']}")
    print("\n")

stream_mode: updates
nội dung: {'model': {'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "SF"}'}, '__gemini_function_call_thought_signatures__': {'call_602578': 'El4KXAERTTIPhGsOcWbJBwIpBzv95zC5VUzHVLwQzuMnhDvt2XuZRp15qhxkw7lTBS9G4GKK5HG+5FpH0TXq9Gn254HKAMH13iWwt8f1yJFsLd/lgRHopWpfX9Yd8FA9'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0663d-8578-7b13-836f-7661554673f5-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'SF'}, 'id': 'call_602578', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 59, 'output_tokens': 16, 'total_tokens': 75, 'input_token_details': {'cache_read': 0}})]}}


stream_mode: custom
nội dung: Đang tra cứu dữ liệu cho thành phố: SF


stream_mode: custom
nội dung: Đã thu thập dữ liệu cho thành phố: SF


stream_mode: updates
nội dung: {'tools': {'messa

## Các pattern phổ biến

Dưới đây là các ví dụ cho thấy các trường hợp sử dụng phổ biến của tính năng streaming.

### Stream các token tư duy / suy luận

Một số mô hình thực hiện việc suy luận nội bộ trước khi đưa ra câu trả lời cuối cùng. Bạn có thể stream các token tư duy / suy luận này ngay khi chúng được tạo ra bằng cách lọc các [standard content block](https://docs.langchain.com/oss/python/langchain/messages#standard-content-blocks) theo `type` là `"reasoning"`.

<div class="alert alert-info">

Đầu ra suy luận phải được kích hoạt trên mô hình.

Xem phần [suy luận](https://docs.langchain.com/oss/python/langchain/models#reasoning) và [trang tích hợp của nhà cung cấp](https://docs.langchain.com/oss/python/integrations/providers/overview) để biết chi tiết về cấu hình.

Để kiểm tra nhanh khả năng hỗ trợ suy luận của một mô hình, hãy xem [models.dev](https://models.dev/).

</div>

Để stream các token tư duy từ một agent, hãy sử dụng `stream_mode="messages"` và lọc ra các content block chứa reasoning:

In [21]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import Runnable


def get_weather(city: str) -> str:
    """Lấy thông tin thời tiết cho một thành phố nhất định."""
    return f"Trời luôn nắng ở {city}!"


model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    timeout=None,
    stop=None,
    thinking_level="high",
)
agent: Runnable = create_agent(
    model=model,
    tools=[get_weather],
)

stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "Thời tiết ở SF như thế nào?"}]},
    version="v3",
)
for message in stream.messages:
    for token in message.reasoning:
        print(f"[suy luận] {token}", end="")
    for token in message.text:
        print(token, end="", flush=True)

Thời tiết ở San Francisco lúc nào cũng có nắng!

Điều này hoạt động tương tự nhau bất kể nhà cung cấp mô hình là ai - LangChain chuẩn hóa các định dạng đặc thù của từng nhà cung cấp (block `thinking` của Anthropic, tóm tắt `reasoning` của OpenAI, v.v.) thành một `type` `"reasoning"` content block tiêu chuẩn thông qua thuộc tính [`content_blocks`](https://docs.langchain.com/oss/python/langchain/messages#standard-content-blocks).

Để stream các token suy luận trực tiếp từ một chat model (mà không cần dùng đến agent), hãy xem bài viết về việc [stream với chat model](https://docs.langchain.com/oss/python/langchain/models#reasoning).

### Stream các tool call

Bạn có thể muốn stream cả hai:

1. JSON một phần khi các [tool call](https://docs.langchain.com/oss/python/langchain/models#tool-calling) đang được tạo ra
2. Các tool call đã hoàn chỉnh, được parse thành công và đã được thực thi

Việc chỉ định [`stream_mode="messages"`](https://docs.langchain.com/oss/python/langchain/streaming#llm-tokens) sẽ stream các [message chunk](https://docs.langchain.com/oss/python/langchain/messages#streaming-and-chunks) tăng dần được tạo ra bởi tất cả các lệnh gọi LLM trong agent. Để truy cập các tin nhắn hoàn chỉnh cùng với các tool call đã được parse:

1. Nếu các tin nhắn đó được theo dõi trong [state](https://docs.langchain.com/oss/python/langchain/short-term-memory) (như trong node model của [`create_agent`](https://docs.langchain.com/oss/python/langchain/agents)), hãy sử dụng `stream_mode=["messages", "updates"]` để truy cập các tin nhắn hoàn chỉnh thông qua [các cập nhật trạng thái](https://docs.langchain.com/oss/python/langchain/streaming#agent-progress) (như được minh họa dưới đây).
2. Nếu các tin nhắn đó không được theo dõi trong state, hãy sử dụng [các bản cập nhật tùy chỉnh](https://docs.langchain.com/oss/python/langchain/streaming#custom-updates) hoặc tổng hợp các chunk trong vòng lặp stream ([xem phần tiếp theo](https://docs.langchain.com/oss/python/langchain/streaming#accessing-completed-messages)).

<div class="alert alert-info">

Tham khảo phần bên dưới về việc [stream từ các sub-agent](https://docs.langchain.com/oss/python/langchain/streaming#streaming-from-sub-agents) nếu agent của bạn bao gồm nhiều LLM.

</div>

In [22]:
from typing import Any

from langchain.agents import create_agent
from langchain.messages import AIMessage, AIMessageChunk, AnyMessage, ToolMessage


def get_weather(city: str) -> str:
    """Lấy thông tin thời tiết cho một thành phố nhất định."""

    return f"Trời luôn nắng ở {city}!"


agent = create_agent("google_genai:gemini-3.5-flash-lite", tools=[get_weather])


def _render_message_chunk(token: AIMessageChunk) -> None:
    if token.text:
        print(token.text, end="|")
    if token.tool_call_chunks:
        print(token.tool_call_chunks)
    # LƯU Ý: mọi nội dung đều có thể truy cập thông qua token.content_blocks


def _render_completed_message(message: AnyMessage) -> None:
    if isinstance(message, AIMessage) and message.tool_calls:
        print(f"Các tool call: {message.tool_calls}")
    if isinstance(message, ToolMessage):
        print(f"Phản hồi tool: {message.content_blocks}")


input_message = {"role": "user", "content": "Thời tiết ở Boston như thế nào?"}
for chunk in agent.stream(
    {"messages": [input_message]},
    stream_mode=["messages", "updates"],
    version="v2",
):
    if chunk["type"] == "messages":
        token, metadata = chunk["data"]
        if isinstance(token, AIMessageChunk):
            _render_message_chunk(token)
    elif chunk["type"] == "updates":
        for source, update in chunk["data"].items():
            if source in ("model", "tools"):  # `source` lưu giữ tên node
                _render_completed_message(update["messages"][-1])

[{'name': 'get_weather', 'args': '{"city": "Boston"}', 'id': 'call_639756', 'index': None, 'type': 'tool_call_chunk'}]
Các tool call: [{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'call_639756', 'type': 'tool_call'}]
Phản hồi tool: [{'type': 'text', 'text': 'Trời luôn nắng ở Boston!'}]
Thời tiết| ở Boston hiện tại: Trời luôn nắng ở Boston!|

#### Truy cập các tin nhắn đã hoàn tất

<div class="alert alert-info">

Nếu các tin nhắn hoàn chỉnh được theo dõi trong [state](https://docs.langchain.com/oss/python/langchain/short-term-memory) của một agent, bạn có thể sử dụng `stream_mode=["messages", "updates"]` như được minh họa trong phần [stream các tool call](https://docs.langchain.com/oss/python/langchain/streaming#streaming-tool-calls) để truy cập các tin nhắn hoàn chỉnh trong quá trình stream.

</div>

Trong một số trường hợp, các tin nhắn hoàn chỉnh không được phản ánh trong [các cập nhật trạng thái](https://docs.langchain.com/oss/python/langchain/streaming#agent-progress). Nếu bạn có quyền truy cập vào các thành phần bên trong của agent, bạn có thể sử dụng [custom updates](https://docs.langchain.com/oss/python/langchain/streaming#custom-updates) để lấy những tin nhắn này khi stream. Nếu không, bạn có thể tổng hợp các message chunk trong vòng lặp stream (xem bên dưới).

Hãy xem xét ví dụ dưới đây, nơi chúng ta tích hợp một [stream writer](https://docs.langchain.com/oss/python/langchain/streaming#custom-updates) vào một [guardrail middleware](https://docs.langchain.com/oss/python/langchain/guardrails#after-agent-guardrails) được tối giản hóa. Middleware này dùng tool call để tạo ra đánh giá cấu trúc "an toàn / không an toàn" (người ta cũng có thể dùng [structured outputs](https://docs.langchain.com/oss/python/langchain/models#structured-output) cho việc này):

In [23]:
from typing import Any, Literal

from langchain.agents.middleware import after_agent, AgentState
from langgraph.runtime import Runtime
from langchain.messages import AIMessage
from langchain.chat_models import init_chat_model
from langgraph.config import get_stream_writer
from pydantic import BaseModel


class ResponseSafety(BaseModel):
    """Đánh giá một phản hồi là an toàn hoặc không an toàn."""
    evaluation: Literal["safe", "unsafe"]


safety_model = init_chat_model("google_genai:gemini-3.5-flash-lite")

@after_agent(can_jump_to=["end"])
def safety_guardrail(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Guardrail dựa trên mô hình: Dùng một LLM để đánh giá độ an toàn của phản hồi."""
    stream_writer = get_stream_writer()
    # Lấy phản hồi của mô hình
    if not state["messages"]:
        return None

    last_message = state["messages"][-1]
    if not isinstance(last_message, AIMessage):
        return None

    # Dùng một mô hình khác để đánh giá độ an toàn
    model_with_tools = safety_model.bind_tools([ResponseSafety], tool_choice="any")
    result = model_with_tools.invoke(
        [
            {
                "role": "system",
                "content": "Đánh giá xem phản hồi AI này nhìn chung là an toàn hay không an toàn."
            },
            {
                "role": "user",
                "content": f"Phản hồi của AI: {last_message.text}"
            }
        ]
    )
    stream_writer(result)

    tool_call = result.tool_calls[0]
    if tool_call["args"]["evaluation"] == "unsafe":
        last_message.content = "Tôi không thể cung cấp câu trả lời đó. Vui lòng định dạng lại yêu cầu của bạn."

    return None

Sau đó, chúng ta có thể tích hợp middleware này vào agent của mình và bao gồm các custom stream event của nó:

In [24]:
from typing import Any

from langchain.agents import create_agent
from langchain.messages import AIMessageChunk, AIMessage, AnyMessage


def get_weather(city: str) -> str:
    """Lấy thông tin thời tiết cho một thành phố nhất định."""

    return f"Trời luôn nắng ở {city}!"


agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[get_weather],
    middleware=[safety_guardrail],
)

def _render_message_chunk(token: AIMessageChunk) -> None:
    if token.text:
        print(token.text, end="|")
    if token.tool_call_chunks:
        print(token.tool_call_chunks)


def _render_completed_message(message: AnyMessage) -> None:
    if isinstance(message, AIMessage) and message.tool_calls:
        print(f"Các tool call: {message.tool_calls}")
    if isinstance(message, ToolMessage):
        print(f"Phản hồi tool: {message.content_blocks}")


input_message = {"role": "user", "content": "Thời tiết ở Boston như thế nào?"}
for chunk in agent.stream(
    {"messages": [input_message]},
    stream_mode=["messages", "updates", "custom"],
    version="v2",
):
    if chunk["type"] == "messages":
        token, metadata = chunk["data"]
        if isinstance(token, AIMessageChunk):
            _render_message_chunk(token)
    elif chunk["type"] == "updates":
        for source, update in chunk["data"].items():
            if source in ("model", "tools"):
                _render_completed_message(update["messages"][-1])
    elif chunk["type"] == "custom":
        # truy cập tin nhắn đã hoàn tất trong luồng stream
        print(f"Các tool call: {chunk['data'].tool_calls}")

[{'name': 'get_weather', 'args': '{"city": "Boston"}', 'id': 'call_711647', 'index': None, 'type': 'tool_call_chunk'}]
Các tool call: [{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'call_711647', 'type': 'tool_call'}]
Phản hồi tool: [{'type': 'text', 'text': 'Trời luôn nắng ở Boston!'}]
Thời tiết| ở Boston hiện tại: Trời luôn nắng ở Boston!|[{'name': 'ResponseSafety', 'args': '{"evaluation": "safe"}', 'id': 'call_632328', 'index': None, 'type': 'tool_call_chunk'}]
Các tool call: [{'name': 'ResponseSafety', 'args': {'evaluation': 'safe'}, 'id': 'call_632328', 'type': 'tool_call'}]


Một cách khác, nếu bạn không thể thêm các custom event vào luồng stream, bạn có thể tổng hợp các message chunk trong vòng lặp stream:

In [25]:
input_message = {"role": "user", "content": "Thời tiết ở Boston như thế nào?"}
full_message = None
for chunk in agent.stream(
    {"messages": [input_message]},
    stream_mode=["messages", "updates"],
    version="v2",
):
    if chunk["type"] == "messages":
        token, metadata = chunk["data"]
        if isinstance(token, AIMessageChunk):
            _render_message_chunk(token)
            full_message = token if full_message is None else full_message + token
            if token.chunk_position == "last":
                if full_message.tool_calls:
                    print(f"Các tool call: {full_message.tool_calls}")
                full_message = None
    elif chunk["type"] == "updates":
        for source, update in chunk["data"].items():
            if source == "tools":
                _render_completed_message(update["messages"][-1])

[{'name': 'get_weather', 'args': '{"city": "Boston"}', 'id': 'call_724700', 'index': None, 'type': 'tool_call_chunk'}]
Các tool call: [{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'call_724700', 'type': 'tool_call'}]
Phản hồi tool: [{'type': 'text', 'text': 'Trời luôn nắng ở Boston!'}]
Thời tiết ở Boston hiện tại: Trời luôn nắng ở Boston!|[{'name': 'ResponseSafety', 'args': '{"evaluation": "safe"}', 'id': 'call_618771', 'index': None, 'type': 'tool_call_chunk'}]
Các tool call: [{'name': 'ResponseSafety', 'args': {'evaluation': 'safe'}, 'id': 'call_618771', 'type': 'tool_call'}]


### Stream với human-in-the-loop

Để xử lý các interrupt kiểu [human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop), chúng ta sẽ mở rộng dựa trên [ví dụ trên](https://docs.langchain.com/oss/python/langchain/streaming#streaming-tool-calls):

1. Cấu hình agent với [human-in-the-loop middleware và một checkpointer](https://docs.langchain.com/oss/python/langchain/human-in-the-loop#configuring-interrupts)
2. Thu thập các lệnh ngắt được sinh ra trong quá trình stream ở chế độ `"updates"`
3. Phản hồi các lệnh ngắt đó bằng một [command](https://docs.langchain.com/oss/python/langchain/human-in-the-loop#responding-to-interrupts)

In [27]:
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import AIMessage, AIMessageChunk, AnyMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, Interrupt


def get_weather(city: str) -> str:
    """Lấy thông tin thời tiết cho một thành phố nhất định."""

    return f"Trời luôn nắng ở {city}!"


checkpointer = InMemorySaver()

agent = create_agent(
    "google_genai:gemini-3.5-flash-lite",
    tools=[get_weather],
    middleware=[
        HumanInTheLoopMiddleware(interrupt_on={"get_weather": True}),
    ],
    checkpointer=checkpointer,
)


def _render_message_chunk(token: AIMessageChunk) -> None:
    if token.text:
        print(token.text, end="|")
    if token.tool_call_chunks:
        print(token.tool_call_chunks)


def _render_completed_message(message: AnyMessage) -> None:
    if isinstance(message, AIMessage) and message.tool_calls:
        print(f"Các tool call: {message.tool_calls}")
    if isinstance(message, ToolMessage):
        print(f"Phản hồi tool: {message.content_blocks}")


def _render_interrupt(interrupt: Interrupt) -> None:
    interrupts = interrupt.value
    for request in interrupts["action_requests"]:
        print(request["description"])


input_message = {
    "role": "user",
    "content": (
        "Bạn có thể tra cứu thời tiết ở Boston và San Francisco không?"
    ),
}
config = {"configurable": {"thread_id": "some_id"}}
interrupts = []
for chunk in agent.stream(
    {"messages": [input_message]},
    config=config,
    stream_mode=["messages", "updates"],
    version="v2",
):
    if chunk["type"] == "messages":
        token, metadata = chunk["data"]
        if isinstance(token, AIMessageChunk):
            _render_message_chunk(token)
    elif chunk["type"] == "updates":
        for source, update in chunk["data"].items():
            if source in ("model", "tools"):
                _render_completed_message(update["messages"][-1])
            if source == "__interrupt__":
                interrupts.extend(update)
                _render_interrupt(update[0])

[{'name': 'get_weather', 'args': '{"city": "Boston"}', 'id': 'call_757509', 'index': None, 'type': 'tool_call_chunk'}]
[{'name': 'get_weather', 'args': '{"city": "San Francisco"}', 'id': 'call_757510', 'index': None, 'type': 'tool_call_chunk'}]
Các tool call: [{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'call_757509', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'call_757510', 'type': 'tool_call'}]
Tool execution requires approval

Tool: get_weather
Args: {'city': 'Boston'}
Tool execution requires approval

Tool: get_weather
Args: {'city': 'San Francisco'}


Tiếp theo, chúng ta thu thập một [quyết định](https://docs.langchain.com/oss/python/langchain/human-in-the-loop#interrupt-decision-types) cho mỗi lệnh ngắt. Quan trọng là thứ tự của các quyết định phải khớp với thứ tự của các hành động mà chúng ta đã thu thập.

Để minh họa, chúng ta sẽ chỉnh sửa một tool call và chấp nhận một tool call khác:

In [28]:
def _get_interrupt_decisions(interrupt: Interrupt) -> list[dict]:
    return [
        {
            "type": "edit",
            "edited_action": {
                "name": "get_weather",
                "args": {"city": "Boston, Vương quốc Anh"},
            },
        }
        if "boston" in request["description"].lower()
        else {"type": "approve"}
        for request in interrupt.value["action_requests"]
    ]

decisions = {}
for interrupt in interrupts:
    decisions[interrupt.id] = {
        "decisions": _get_interrupt_decisions(interrupt)
    }

decisions

{'78e0de26bdb3b730a6acbef46a4040ce': {'decisions': [{'type': 'edit',
    'edited_action': {'name': 'get_weather',
     'args': {'city': 'Boston, Vương quốc Anh'}}},
   {'type': 'approve'}]}}

Sau đó, chúng ta có thể tiếp tục bằng cách truyền một [command](https://docs.langchain.com/oss/python/langchain/human-in-the-loop#responding-to-interrupts) vào cùng vòng lặp stream đó:

In [29]:
interrupts = []
for chunk in agent.stream(
    Command(resume=decisions),
    config=config,
    stream_mode=["messages", "updates"],
    version="v2",
):
    # Vòng lặp stream không thay đổi
    if chunk["type"] == "messages":
        token, metadata = chunk["data"]
        if isinstance(token, AIMessageChunk):
            _render_message_chunk(token)
    elif chunk["type"] == "updates":
        for source, update in chunk["data"].items():
            if source in ("model", "tools"):
                _render_completed_message(update["messages"][-1])
            if source == "__interrupt__":
                interrupts.extend(update)
                _render_interrupt(update[0])

Phản hồi tool: [{'type': 'text', 'text': 'Trời luôn nắng ở Boston, Vương quốc Anh!'}]
Phản hồi tool: [{'type': 'text', 'text': 'Trời luôn nắng ở San Francisco!'}]
Thời tiết hiện tại ở cả **Boston** và **San Francisco** đều đang rất| đẹp: trời luôn nắng ở cả hai thành phố!|

### Stream từ các sub-agent

Khi có nhiều LLM tại bất kỳ thời điểm nào trong một agent, thường cần phải làm rõ nguồn gốc của các tin nhắn ngay khi chúng được tạo ra.

Để thực hiện việc này, hãy truyền thuộc tính [`name`](https://reference.langchain.com/python/langchain/agents/#langchain.agents.create_agent\(name\)) cho từng agent khi khởi tạo. Tên này sau đó sẽ khả dụng trong metadata qua key `lc_agent_name` khi stream ở chế độ `"messages"`.

Dưới đây, chúng tôi cập nhật lại ví dụ [stream các tool call](https://docs.langchain.com/oss/python/langchain/streaming#streaming-tool-calls):

1. Chúng tôi thay thế tool ban đầu bằng một tool `call_weather_agent` có khả năng gọi ngầm một agent ở bên trong
2. Thêm một cái `name` cho mỗi agent
3. Chỉ định [`subgraphs=True`](https://docs.langchain.com/oss/python/langgraph/use-subgraphs#stream-subgraph-outputs) khi tạo stream
4. Quá trình xử lý stream giống hệt như trước, nhưng chúng ta bổ sung thêm logic để theo dõi xem agent nào đang hoạt động bằng cách sử dụng tham số `name` của `create_agent`.

<div class="alert alert-info">

Khi bạn thiết lập thuộc tính `name` trên một agent, tên đó cũng sẽ được đính kèm vào bất kỳ `AIMessage` nào được tạo bởi agent đó.

</div>

Đầu tiên, chúng ta cấu hình agent:

In [30]:
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage, AnyMessage


def get_weather(city: str) -> str:
    """Lấy thông tin thời tiết cho một thành phố nhất định."""

    return f"Trời luôn nắng ở {city}!"


weather_model = init_chat_model("google_genai:gemini-3.5-flash-lite")
weather_agent = create_agent(
    model=weather_model,
    tools=[get_weather],
    name="weather_agent",
)


def call_weather_agent(query: str) -> str:
    """Truy vấn agent thời tiết."""
    result = weather_agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return result["messages"][-1].text


supervisor_model = init_chat_model("google_genai:gemini-3.5-flash-lite")
agent = create_agent(
    model=supervisor_model,
    tools=[call_weather_agent],
    name="supervisor",
)

Tiếp theo, chúng ta thêm logic vào vòng lặp stream để báo cáo agent nào đang phát ra các token:

In [31]:
def _render_message_chunk(token: AIMessageChunk) -> None:
    if token.text:
        print(token.text, end="|")
    if token.tool_call_chunks:
        print(token.tool_call_chunks)


def _render_completed_message(message: AnyMessage) -> None:
    if isinstance(message, AIMessage) and message.tool_calls:
        print(f"Các tool call: {message.tool_calls}")
    if isinstance(message, ToolMessage):
        print(f"Phản hồi tool: {message.content_blocks}")


input_message = {"role": "user", "content": "Thời tiết ở Boston như thế nào?"}
current_agent = None
for chunk in agent.stream(
    {"messages": [input_message]},
    stream_mode=["messages", "updates"],
    subgraphs=True,
    version="v2",
):
    if chunk["type"] == "messages":
        token, metadata = chunk["data"]
        if agent_name := metadata.get("lc_agent_name"):
            if agent_name != current_agent:
                print(f"🤖 {agent_name}: ")
                current_agent = agent_name
        if isinstance(token, AIMessageChunk):
            _render_message_chunk(token)
    elif chunk["type"] == "updates":
        for source, update in chunk["data"].items():
            if source in ("model", "tools"):
                _render_completed_message(update["messages"][-1])

🤖 supervisor: 
[{'name': 'call_weather_agent', 'args': '{"query": "Th\\u1eddi ti\\u1ebft \\u1edf Boston nh\\u01b0 th\\u1ebf n\\u00e0o?"}', 'id': 'call_434182', 'index': None, 'type': 'tool_call_chunk'}]
Các tool call: [{'name': 'call_weather_agent', 'args': {'query': 'Thời tiết ở Boston như thế nào?'}, 'id': 'call_434182', 'type': 'tool_call'}]
🤖 weather_agent: 
[{'name': 'get_weather', 'args': '{"city": "Boston"}', 'id': 'call_770525', 'index': None, 'type': 'tool_call_chunk'}]
Các tool call: [{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'call_770525', 'type': 'tool_call'}]
Phản hồi tool: [{'type': 'text', 'text': 'Trời luôn nắng ở Boston!'}]
Thời tiết| ở Boston hiện tại: Trời luôn nắng ở Boston!|🤖 supervisor: 
Phản hồi tool: [{'type': 'text', 'text': 'Thời tiết ở Boston hiện tại: Trời luôn nắng ở Boston!'}]
Thời tiết ở Boston hiện tại: Trời luôn nắng ở Boston!|

## Vô hiệu hóa streaming

Trong một số ứng dụng, bạn có thể cần vô hiệu hóa luồng stream các token riêng lẻ đối với một mô hình cụ thể. Điều này hữu ích khi:

* Làm việc với các hệ thống [multi-agent](https://docs.langchain.com/oss/python/langchain/multi-agent) để kiểm soát agent nào được quyền stream đầu ra của chúng
* Kết hợp các mô hình có hỗ trợ streaming với các mô hình không hỗ trợ
* Triển khai lên [LangSmith](https://docs.langchain.com/langsmith/observability) và muốn ngăn chặn một số đầu ra cụ thể của mô hình bị stream đến client

Thiết lập `streaming=False` khi khởi tạo mô hình.

```python
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-5.5",
    streaming=False
)
```

<div class="alert alert-success">

Khi triển khai lên LangSmith, hãy đặt `streaming=False` đối với bất kỳ mô hình nào mà bạn không muốn stream đầu ra của nó cho client. Việc này được cấu hình trong mã graph của bạn trước khi triển khai.

</div>

<div class="alert alert-info">

Không phải tất cả các bản tích hợp chat model đều hỗ trợ tham số `streaming`. Nếu mô hình của bạn không hỗ trợ tham số này, hãy sử dụng `disable_streaming=True` để thay thế. Tham số này khả dụng trên mọi chat model thông qua base class.

</div>

Xem [hướng dẫn về LangGraph streaming](https://docs.langchain.com/oss/python/langgraph/streaming#disable-streaming-for-specific-chat-models) để biết thêm chi tiết.

## Định dạng streaming v2

<div class="alert alert-info">

Yêu cầu LangGraph >= 1.1.

</div>

Truyền tham số `version="v2"` vào hàm `stream()` hoặc `astream()` để nhận về một định dạng đầu ra thống nhất. Mỗi chunk đều là một dict `StreamPart` với các key `type`, `ns` và `data` - chung một cấu trúc bất kể chế độ stream hay số lượng chế độ được dùng:

In [32]:
# Định dạng thống nhất - không còn cần giải nén tuple nữa
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "Thời tiết ở SF như thế nào?"}]},
    stream_mode=["updates", "custom"],
    version="v2",
):
    print(chunk["type"])  # "updates" hoặc "custom"
    print(chunk["data"])  # payload

updates
{'model': {'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'call_weather_agent', 'arguments': '{"query": "San Francisco"}'}, '__gemini_function_call_thought_signatures__': {'call_968932': 'El4KXAERTTIPKmJm2quDvGeEGl/wJIFhkkm/Qae9k/wGP0XewY6fE9LV3FQo8oxYqigo5IITRhRp87K41oqi3LXuG2mcl3Ll5D8XKsAfTImCELG7pT3GUCNoLsV6my6v'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, name='supervisor', id='lc_run--01a066ac-72c0-7d90-9f97-2d4370bbe0b0-0', tool_calls=[{'name': 'call_weather_agent', 'args': {'query': 'San Francisco'}, 'id': 'call_968932', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 55, 'output_tokens': 19, 'total_tokens': 74, 'input_token_details': {'cache_read': 0}})]}}
updates
{'tools': {'messages': [ToolMessage(content='Thời tiết ở San Francisco hiện tại: Trời luôn nắng ở San Francisco!', name='call_weather_agent', id='0bd

Định dạng v2 cũng cải tiến hàm `invoke()` - nó trả về một đối tượng `GraphOutput` với các thuộc tính `.value` và `.interrupts`, tách biệt rõ ràng state khỏi interrupt metadata:

In [33]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Xin chào"}]},
    version="v2",
)
print(result.value)       # trạng thái (dict, Pydantic model, hoặc dataclass)
print(result.interrupts)  # tuple chứa các đối tượng Interrupt (rỗng nếu không có)

{'messages': [HumanMessage(content='Xin chào', additional_kwargs={}, response_metadata={}, id='288f9398-dd7f-4456-b97f-6e85ec087243'), AIMessage(content=[{'type': 'text', 'text': 'Chào bạn! Tôi có thể giúp gì cho bạn hôm nay?', 'extras': {'signature': 'El4KXAERTTIP8mPaB96Giv13b65kpkGX5mecbxmmiomXdJtQ3CJx31V5wkobf3WJ6Ptn9wVHEsJQbmDAy94AcsSJZ7wqRjK75XNJgkTKc98EvzoR0Mri/YI2avHT7wij'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, name='supervisor', id='lc_run--01a066ad-5e00-7f63-a760-23b574760e36-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 13, 'total_tokens': 62, 'input_token_details': {'cache_read': 0}})]}
()


Xem [tài liệu về LangGraph streaming](https://docs.langchain.com/oss/python/langgraph/streaming#stream-output-format-v2) để biết thêm chi tiết về định dạng v2, bao gồm type narrowing, ánh xạ về Pydantic/dataclass và subgraph streaming.

## Bài viết liên quan

* [Frontend streaming](https://docs.langchain.com/oss/python/langchain/frontend/overview) - Xây dựng giao diện React với [`useStream`](https://reference.langchain.com/javascript/langchain-react/index/useStream) để tương tác trực tiếp theo thời gian thực với agent
* [Streaming với các chat model](https://docs.langchain.com/oss/python/langchain/models#stream) - Stream các token trực tiếp từ một chat model mà không cần sử dụng agent hoặc graph
* [Suy luận với các chat model](https://docs.langchain.com/oss/python/langchain/models#reasoning) - Cấu hình và truy cập đầu ra suy luận từ các chat model
* [Các content block tiêu chuẩn](https://docs.langchain.com/oss/python/langchain/messages#standard-content-blocks) - Tìm hiểu định dạng content block chuẩn hóa được sử dụng cho quá trình suy luận, văn bản và các loại nội dung khác
* [Streaming với human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop#streaming-with-human-in-the-loop) - Stream tiến trình của agent đồng thời xử lý các lệnh ngắt chờ con người phê duyệt
* [LangGraph streaming](https://docs.langchain.com/oss/python/langgraph/streaming) - Các tùy chọn streaming nâng cao bao gồm `values`, chế độ `debug` và stream các subgraph